In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
from sklearn.feature_selection import mutual_info_classif
from pathlib import Path

In [55]:
#collect the data
columns = [
    "Profile_mean",
    "Profile_stdev",
    "Profile_skewness",
    "Profile_kurtosis",
    "DM_mean",
    "DM_stdev",
    "DM_skewness",
    "DM_kurtosis",
    "class"
]

df = pd.read_csv(
    r"..\data\htru2\HTRU_2.csv",
    header=None,
    names=columns
)

df.head()

,Profile_mean,Profile_stdev,Profile_skewness,Profile_kurtosis,DM_mean,DM_stdev,DM_skewness,DM_kurtosis,class
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225,0
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580,0
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909,0
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661,0
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306,0


In [56]:
#basic data information
print(df.info())
print(df.describe())
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17898 entries, 0 to 17897
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Profile_mean      17898 non-null  float64
 1   Profile_stdev     17898 non-null  float64
 2   Profile_skewness  17898 non-null  float64
 3   Profile_kurtosis  17898 non-null  float64
 4   DM_mean           17898 non-null  float64
 5   DM_stdev          17898 non-null  float64
 6   DM_skewness       17898 non-null  float64
 7   DM_kurtosis       17898 non-null  float64
 8   class             17898 non-null  int64  
dtypes: float64(8), int64(1)
memory usage: 1.2 MB
None
       Profile_mean  Profile_stdev  Profile_skewness  Profile_kurtosis  \
count  17898.000000   17898.000000      17898.000000      17898.000000   
mean     111.079968      46.549532          0.477857          1.770279   
std       25.652935       6.843189          1.064040          6.167913   
min        5.812500     

In [57]:
#check null values
print(df.isnull().sum())

Profile_mean        0
Profile_stdev       0
Profile_skewness    0
Profile_kurtosis    0
DM_mean             0
DM_stdev            0
DM_skewness         0
DM_kurtosis         0
class               0
dtype: int64


In [58]:
#check for duplicates
print(df.duplicated().sum())

0


In [59]:
#check extreme large values
print((df.drop(columns=["class"]) > 1e6).sum())

Profile_mean        0
Profile_stdev       0
Profile_skewness    0
Profile_kurtosis    0
DM_mean             0
DM_stdev            0
DM_skewness         0
DM_kurtosis         0
dtype: int64


In [60]:
#check extreme small values
print((df.drop(columns=["class"]) < -1e6).sum())

Profile_mean        0
Profile_stdev       0
Profile_skewness    0
Profile_kurtosis    0
DM_mean             0
DM_stdev            0
DM_skewness         0
DM_kurtosis         0
dtype: int64


In [61]:
print(df["class"].value_counts())
print(df["class"].value_counts(normalize=True))
# dataset is imbalanced as shown from result

class
0    16259
1     1639
Name: count, dtype: int64
class
0    0.908426
1    0.091574
Name: proportion, dtype: float64


In [62]:
# check weather there are zeros in features
print((df.drop(columns=["class"]) == 0).sum())

Profile_mean        0
Profile_stdev       0
Profile_skewness    0
Profile_kurtosis    0
DM_mean             0
DM_stdev            0
DM_skewness         0
DM_kurtosis         0
dtype: int64


In [63]:
# feature engineering

df_engineered = df.copy()

df_engineered["profile_cv"] = df_engineered["Profile_stdev"] / (df_engineered["Profile_mean"] + 1e-6)
df_engineered["DM_cv"] = df_engineered["DM_stdev"] / (df_engineered["DM_mean"] + 1e-6)
df_engineered["profile_shape_score"] = abs(df_engineered["Profile_skewness"]) + abs(df_engineered["Profile_kurtosis"])
df_engineered["DM_shape_score"] = abs(df_engineered["DM_skewness"]) + abs(df_engineered["DM_kurtosis"])


In [64]:
#check for missing values
print(df_engineered.isnull().sum())


Profile_mean           0
Profile_stdev          0
Profile_skewness       0
Profile_kurtosis       0
DM_mean                0
DM_stdev               0
DM_skewness            0
DM_kurtosis            0
class                  0
profile_cv             0
DM_cv                  0
profile_shape_score    0
DM_shape_score         0
dtype: int64


In [65]:
#check infinite values
print(np.isinf(df_engineered).sum())

Profile_mean           0
Profile_stdev          0
Profile_skewness       0
Profile_kurtosis       0
DM_mean                0
DM_stdev               0
DM_skewness            0
DM_kurtosis            0
class                  0
profile_cv             0
DM_cv                  0
profile_shape_score    0
DM_shape_score         0
dtype: int64


In [66]:
#check extreme large values
print((df_engineered.drop(columns=["class"]) > 1e6).sum())

Profile_mean           0
Profile_stdev          0
Profile_skewness       0
Profile_kurtosis       0
DM_mean                0
DM_stdev               0
DM_skewness            0
DM_kurtosis            0
profile_cv             0
DM_cv                  0
profile_shape_score    0
DM_shape_score         0
dtype: int64


In [67]:
# check extreme small values
print((df_engineered.drop(columns=["class"]) < -1e6).sum())

Profile_mean           0
Profile_stdev          0
Profile_skewness       0
Profile_kurtosis       0
DM_mean                0
DM_stdev               0
DM_skewness            0
DM_kurtosis            0
profile_cv             0
DM_cv                  0
profile_shape_score    0
DM_shape_score         0
dtype: int64


In [68]:
#70% for training, 15% for development, and 15% for testing
x_engineered = df_engineered.drop(columns=["class"])
y_engineered = df_engineered["class"]

x_engineered_temp, x_engineered_test, y_engineered_temp, y_engineered_test = train_test_split(
    x_engineered,
    y_engineered,
    test_size=0.15,
    stratify=y_engineered,
    random_state=7
)

x_engineered_train, x_engineered_dev, y_engineered_train, y_engineered_dev = train_test_split(
    x_engineered_temp,
    y_engineered_temp,
    test_size=0.15/0.85,
    stratify=y_engineered_temp,
    random_state=7
)

train_engineered = x_engineered_train.copy()
train_engineered["class"] = y_engineered_train

dev_engineered = x_engineered_dev.copy()
dev_engineered["class"] = y_engineered_dev

test_engineered = x_engineered_test.copy()
test_engineered["class"] = y_engineered_test



# Save datasets
train_engineered.to_csv("../outputs/processed_data/train_engineered.csv", index=False)
dev_engineered.to_csv("../outputs/processed_data/dev_engineered.csv", index=False)
test_engineered.to_csv("../outputs/processed_data/test_engineered.csv", index=False)

In [69]:
x_engineered_train.shape, y_engineered_train.shape, x_engineered_dev.shape, y_engineered_dev.shape, x_engineered_test.shape, y_engineered_test.shape

((12528, 12), (12528,), (2685, 12), (2685,), (2685, 12), (2685,))

In [70]:
mi_scores = mutual_info_classif(
    x_engineered_train,
    y_engineered_train,
    random_state=7
)

mi_results = pd.DataFrame({
    "feature": x_engineered_train.columns,
    "mi_score": mi_scores
}).sort_values(by="mi_score", ascending=False)

mi_results

,feature,mi_score
2,Profile_skewness,0.225183
10,profile_shape_score,0.205575
3,Profile_kurtosis,0.195685
0,Profile_mean,0.189617
8,profile_cv,0.169137
11,DM_shape_score,0.121295
5,DM_stdev,0.120651
7,DM_kurtosis,0.114665
4,DM_mean,0.114011
6,DM_skewness,0.113745
